# symmetry-neurons on Colab

Prune the parameters a multimodal LLM activates for grade-school math (GSM8K) but not for reading comprehension (RACE), then test whether it can still spot the odd shape out in a grid of polygons. See the [README](https://github.com/lcote-math/symmetry-neurons) for the background.

**Before you start:**
1. Choose **Runtime → Change runtime type → GPU**.
2. Gemma models are gated: accept the license on the [model page](https://huggingface.co/google/gemma-3-12b-it) with your Hugging Face account. Then add a [Hugging Face token](https://huggingface.co/settings/tokens) as a Colab secret named `HF_TOKEN` (key icon in the left sidebar), or log in with the cell below.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Setup

In [ ]:
!git clone -q https://github.com/lcote-math/symmetry-neurons.git
%cd symmetry-neurons
!pip install -q -r requirements.txt

In [ ]:
# Skip this cell if you added HF_TOKEN as a Colab secret.
from huggingface_hub import notebook_login
notebook_login()

## Choose a model and sample size

Gemma 3 12B (used in the blog post) needs about 24 GB of GPU memory, so it needs an A100 or L4 runtime (Colab Pro). Gemma 3 4B fits on the free T4.

In [ ]:
MODEL = "google/gemma-3-4b-it"   # or "google/gemma-3-12b-it" on an A100 / L4
N = 20                           # samples per dataset (20 in the blog post)
SEED = 42                        # change for a different random sample
BATCH_SIZE = 4                   # lower this if you run out of GPU memory

OUT = f"outputs/{MODEL.split('/')[-1]}_n{N}_seed{SEED}"

## Build the datasets and look at an example

In [ ]:
!python -m src.make_datasets gsm8k race regular_c4 irregular_c4 --n {N} --seed {SEED}

In [ ]:
import pandas as pd
from IPython.display import Image, display
from src.make_datasets import dataset_path

example = pd.read_csv(dataset_path("regular_c4", N, SEED)).iloc[0]
print(example.question)
print("Correct answer:", example.intruder_position)
display(Image(example.image_path, width=400))

## Run the experiment

In [ ]:
!python main.py --model_name {MODEL} --n {N} --seed {SEED} --batch_size {BATCH_SIZE} --output_dir {OUT}

## Inspect the answers

The `correct` column uses a heuristic: the last position mentioned in the answer. Read the answers before trusting the scores.

In [ ]:
pd.set_option("display.max_colwidth", 200)
for stage in ["pre_pruning", "post_pruning"]:
    for task in ["regular_c4", "irregular_c4"]:
        df = pd.read_csv(f"{OUT}/{stage}_{task}.csv")
        print(f"{stage} {task}: {df.correct.sum()}/{len(df)}")
        display(df[["intruder_position", "predicted_position", "correct", "model_answer"]].head())

In [ ]:
# Optional: download the outputs
!zip -qr outputs.zip {OUT}
from google.colab import files
files.download("outputs.zip")